<div style="background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); color: white; padding: 25px; text-align: center; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.2); margin: 20px 0;">

# Demo.2 - 手势「2」识别：掌心圆算法

本 Notebook 演示一种**无需手势分类模型**、仅凭手部关键点几何关系判断手势的方法。

</div>

<div style="background-color: #eef6ff; border-left: 4px solid #2563eb; padding: 15px; border-radius: 4px; margin-top: 12px;">

## 本节你将学会

1. 用掌心圆近似表示手掌区域。
2. 通过“指尖是否跑出掌心圆”判断手指伸出或收拢。
3. 用几何规则识别 ✌️ 手势，而不是依赖额外分类模型。

**建议课堂观察**：把食指、中指、三指张开、握拳这几种手势轮流做出来，比对 outside / inside 列表如何变化。

</div>

### 课堂观察任务

这个实验的核心不是“识别出 2”，而是理解**为什么几何规则可以替代分类模型**。请先带着下面的问题去看：

1. 掌心圆为什么可以近似表示手掌区域？
2. 为什么只看“指尖是否在圆外”还不够，还要比较指尖和前一关节的距离？
3. 哪些手势最容易和 ✌️ 混淆，比如三指张开、拇指半伸、手掌倾斜？

建议同学把 outside / inside 的变化记下来，再去解释模型为什么判成“Gesture 2”或“Not gesture 2”。

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 0. 环境安装

首次运行，请执行以下安装命令

</div>

### 弹窗显示的重要说明

如果你希望使用 `cv2.imshow(...)` 打开 OpenCV 弹窗，当前环境里**不能同时混装多个 OpenCV 版本或 headless 版本**。

如果同时装了 `opencv-python-headless`，OpenCV 往往会失去窗口能力，表现为：

- `cv2.imshow` 报错
- `cv2.destroyAllWindows` 报错
- 笔记本只能退回到内联预览模式

因此下面的安装单元采用“先卸载冲突包，再安装带 GUI 的版本”的方式。安装完成后，请**重启内核**再继续运行。

In [1]:
# %pip uninstall -y opencv-python-headless opencv-python opencv-contrib-python opencv-contrib-python-headless
%pip install mediapipe opencv-contrib-python==4.13.0.92 numpy

Note: you may need to restart the kernel to use updated packages.


### 安装后必须执行这一步

如果上面的安装单元执行完成，请先**重启内核**，再从“导入依赖库”开始重新运行。

这是为了确保 notebook 重新加载新的 OpenCV 版本，而不是继续使用旧内存中的 `cv2`。

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 1. 导入依赖库

</div>

In [2]:
import time
from pathlib import Path

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision
import numpy as np

print("依赖库导入成功 ✓")

依赖库导入成功 ✓


<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 2. 常量配置

- `FINGER_NAMES`：五根手指名称，顺序与 MediaPipe 关键点索引一致
- `TIP_IDS`：五个指尖的关键点索引（4=拇指尖，8=食指尖，以此类推）

</div>

In [3]:
WINDOW_NAME = "Demo.2 - Gesture Two Detection"
MODEL_PATH = Path("models/hand_landmarker.task")

FINGER_NAMES = ["thumb", "index", "middle", "ring", "pinky"]
TIP_IDS = [4, 8, 12, 16, 20]  # 各指尖的关键点索引

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"未找到手部模型文件: {MODEL_PATH}")

for name, tip_id in zip(FINGER_NAMES, TIP_IDS):
    print(f"  {name:8s} → 指尖关键点索引: {tip_id}")
print("课堂提示：阈值 1.18 不是唯一正确答案，而是一个经验值，可以鼓励同学课后微调比较。")

  thumb    → 指尖关键点索引: 4
  index    → 指尖关键点索引: 8
  middle   → 指尖关键点索引: 12
  ring     → 指尖关键点索引: 16
  pinky    → 指尖关键点索引: 20
课堂提示：阈值 1.18 不是唯一正确答案，而是一个经验值，可以鼓励同学课后微调比较。


<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 3. 坐标转换

MediaPipe 返回的关键点坐标是 **归一化坐标**（值域 0~1），转换为 numpy 像素坐标数组。

</div>

In [4]:
def to_pixel(landmark, width, height):
    """归一化坐标 → numpy int32 像素坐标数组。"""
    return np.array([int(landmark.x * width), int(landmark.y * height)], dtype=np.int32)

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 4. 掌心圆计算

`cv2.minEnclosingCircle` 计算一组点的最小外接圆。选取 6 个锚点（0,1,5,9,13,17）分布在手腕和掌骨关节，描述手掌大小和位置。

</div>

In [5]:
def compute_palm_circle(hand_landmarks, width, height):
    """
    用手腕 + 5 个掌骨关节计算最小外接圆。
    返回: (palm_points, center, radius)
    """
    palm_indices = [0, 1, 5, 9, 13, 17]  # 手腕 + 各指掌骨关节
    palm_points = np.array(
        [to_pixel(hand_landmarks[idx], width, height) for idx in palm_indices],
        dtype=np.int32,
    )
    # minEnclosingCircle 需要 float32 输入
    (cx, cy), radius = cv2.minEnclosingCircle(palm_points.astype(np.float32))
    return palm_points, np.array([cx, cy], dtype=np.float32), float(radius)

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 5. 手指伸出/收拢判断

对每根手指：计算指尖到掌心圆心的距离，若 **距离 > 半径 × 1.18** 且 **指尖比关节更远**，则判定「伸出」。

</div>

In [6]:
def finger_inside_outside(hand_landmarks, center, radius, width, height):
    """
    判断每根手指是伸出(outside)还是收拢(inside)。
    返回: (inside列表, outside列表, finger_info列表)
    finger_info 每项: (手指名, 指尖像素坐标, 是否伸出)
    """
    inside = []
    outside = []
    finger_info = []

    for name, tip_idx in zip(FINGER_NAMES, TIP_IDS):
        tip = to_pixel(hand_landmarks[tip_idx], width, height).astype(np.float32)
        dip = to_pixel(hand_landmarks[max(tip_idx - 1, 0)], width, height).astype(np.float32)
        distance_tip = np.linalg.norm(tip - center)
        distance_dip = np.linalg.norm(dip - center)
        # 双重条件：超出圆外 + 指尖比关节更远
        is_outside = distance_tip > radius * 1.18 and distance_tip > distance_dip
        if is_outside:
            outside.append(name)
        else:
            inside.append(name)
        finger_info.append((name, tip.astype(np.int32), is_outside))

    return inside, outside, finger_info

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 6. 可视化绘制

绿色圆点 = 手指伸出；蓝色圆点 = 手指收拢。同时绘制掌心外接圆和凸包轮廓。

</div>

In [7]:
def draw_overlay(image, palm_points, center, radius, finger_info, fps):
    """绘制掌心圆、凸包和指尖状态标注。"""
    if len(palm_points) >= 3 and radius > 1:
        hull = cv2.convexHull(palm_points)
        cv2.polylines(image, [hull], True, (60, 230, 255), 2, cv2.LINE_AA)  # 凸包轮廓
        cv2.circle(image, tuple(center.astype(np.int32)), int(radius), (60, 230, 255), 2, cv2.LINE_AA)  # 掌心圆

        for point in palm_points:
            cv2.circle(image, tuple(point), 8, (255, 255, 255), -1, cv2.LINE_AA)
            cv2.circle(image, tuple(point), 4, (80, 180, 255), -1, cv2.LINE_AA)

    for name, tip, is_outside in finger_info:
        color = (80, 230, 120) if is_outside else (120, 120, 255)  # 绿=伸出, 蓝=收拢
        text = f"{name}: outside" if is_outside else f"{name}: inside"
        cv2.circle(image, tuple(tip), 12, color, 2, cv2.LINE_AA)
        cv2.putText(
            image, text, (tip[0] + 8, tip[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.52, color, 2, cv2.LINE_AA,
        )

    overlay = image.copy()
    cv2.rectangle(overlay, (18, 18), (510, 150), (18, 28, 44), -1)
    cv2.addWeighted(overlay, 0.45, image, 0.55, 0, image)
    cv2.putText(
        image, "Gesture 2 strategy: palm circle + finger outside/inside", (32, 55),
        cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 245, 225), 2, cv2.LINE_AA,
    )
    cv2.putText(
        image, f"FPS: {fps:.1f}", (32, 90),
        cv2.FONT_HERSHEY_SIMPLEX, 0.68, (120, 240, 255), 2, cv2.LINE_AA,
    )
    cv2.putText(
        image, "Press Q or ESC to quit", (32, 125),
        cv2.FONT_HERSHEY_SIMPLEX, 0.62, (220, 230, 240), 2, cv2.LINE_AA,
    )

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 7. 主程序：手势识别循环

### 关键步骤
1. **摄像头采集**：`cv2.VideoCapture(0)` 打开默认摄像头。
2. **掌心圆计算**：取 6 个锚点计算最小外接圆。
3. **手指判断**：指尖距离 > 半径 × 1.18 且比前一关节更远。
4. **手势识别**：`set(outside) == {"index", "middle"}` → 手势 2。

### 课堂上建议同学重点观察
1. 为什么只伸出食指和中指时，outside 列表会更稳定？
2. 当拇指半伸半收时，这套规则为什么容易出错？
3. 如果把 1.18 改得更大或更小，会发生什么？

**运行后**：比出 ✌️ 手势测试识别效果，按 `Q` 退出。

</div>

### 运行前，先想清楚这套规则什么时候会失效

建议先不要急着比 ✌️，而是先预测几种边界情况：

- 如果拇指没有完全收回，这套规则会不会误判？
- 如果手掌明显倾斜，掌心圆会不会失真？
- 如果阈值 `1.18` 改成 `1.05` 或 `1.30`，outside 列表会变得更宽松还是更严格？

运行时可以按“握拳✊ → ✌️ → 三指张开👌 → 点赞👍”这个顺序测试，观察 outside / inside 列表怎样变化。

In [10]:
def main():
    from IPython.display import clear_output, display
    from PIL import Image

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("无法打开摄像头 0。")

    def highgui_available():
        try:
            cv2.namedWindow('highgui_test', cv2.WINDOW_NORMAL)
            cv2.destroyWindow('highgui_test')
            return True
        except cv2.error:
            return False

    use_highgui = highgui_available()
    max_preview_frames = 120
    frame_counter = 0
    prev_time = time.time()

    if use_highgui:
        print("已启用 OpenCV 窗口模式，按 Q 或 ESC 退出。")
    else:
        print("当前环境不支持 cv2.imshow，已切换为 notebook 内联预览模式。")
        print(f"内联模式将自动预览 {max_preview_frames} 帧。")

    options = mp_vision.HandLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=mp_vision.RunningMode.VIDEO,
        num_hands=1,
        min_hand_detection_confidence=0.6,
        min_hand_presence_confidence=0.6,
        min_tracking_confidence=0.6,
    )
    with mp_vision.HandLandmarker.create_from_options(options) as landmarker:
        while True:
            success, frame = cap.read()
            if not success:
                print("读取摄像头帧失败。")
                break

            frame = cv2.flip(frame, 1)
            height, width = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int(time.time() * 1000)
            results = landmarker.detect_for_video(mp_image, timestamp_ms)
            canvas = frame.copy()

            current_time = time.time()
            fps = 1.0 / max(current_time - prev_time, 1e-6)
            prev_time = current_time

            if results.hand_landmarks:
                hand_landmarks = results.hand_landmarks[0]
                palm_points, center, radius = compute_palm_circle(hand_landmarks, width, height)
                inside, outside, finger_info = finger_inside_outside(
                    hand_landmarks, center, radius, width, height,
                )
                draw_overlay(canvas, palm_points, center, radius, finger_info, fps)

                # 核心判断：伸出的手指集合恰好等于 {食指, 中指}
                is_gesture_two = set(outside) == {"index", "middle"}
                status_text = "Gesture 2 detected" if is_gesture_two else "Not gesture 2"
                status_color = (80, 240, 120) if is_gesture_two else (90, 170, 255)
                cv2.putText(
                    canvas, status_text, (32, 470),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, status_color, 3, cv2.LINE_AA,
                )
                cv2.putText(
                    canvas, f"Outside: {', '.join(outside) if outside else 'none'}", (32, 510),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA,
                )
                cv2.putText(
                    canvas, f"Inside: {', '.join(inside) if inside else 'none'}", (32, 545),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (210, 220, 235), 2, cv2.LINE_AA,
                )
            else:
                draw_overlay(
                    canvas,
                    np.array([[0, 0], [0, 0], [0, 0]], dtype=np.int32),
                    np.array([0, 0], dtype=np.float32),
                    0.0, [], fps,
                )
                cv2.putText(
                    canvas, "Show one hand to the camera", (32, 500),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.95, (90, 170, 255), 3, cv2.LINE_AA,
                )

            if use_highgui:
                cv2.imshow(WINDOW_NAME, canvas)
                key = cv2.waitKey(1) & 0xFF
                if key in (ord("q"), 27):
                    break
            else:
                canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
                clear_output(wait=True)
                display(Image.fromarray(canvas_rgb))
                time.sleep(0.05)
                frame_counter += 1
                if frame_counter >= max_preview_frames:
                    print("内联预览已结束。若要继续观察，请重新运行本单元。")
                    break

    cap.release()
    if use_highgui:
        try:
            cv2.destroyAllWindows()
            cv2.waitKey(1)
        except cv2.error:
            pass


main()

已启用 OpenCV 窗口模式，按 Q 或 ESC 退出。


### 结果解读与排错提示

这套方法的优点是轻量、直观，但也有边界条件：

1. 只用几何规则时，复杂手势容易互相混淆。
2. 手掌倾斜太大或部分手指被遮挡时，掌心圆会失真。
3. 如果画面中同时出现多只手，本例只取第一只手做判断。

建议同学课后尝试：把目标手势从 ✌️ 改成“点赞👍”或“OK👌”，思考需要新增哪些几何规则。